In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =========================================================
# User settings
# =========================================================
MEASURED_CSV = "data/HIV_TAR_measured_cs.csv"
N_CONFORMERS = 20
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

ENSEMBLE_FILES = {
    'FARFAR'    : 'data/merged_shifts_wide_FARFAR.csv',
    'Alphafold' : 'data/merged_shifts_wide_Alphafold.csv',
    'cMD'       : 'data/merged_shifts_wide_cMD.csv',
    'GaMD'      : 'data/merged_shifts_wide_GaMD.csv',
    'REST2'     : 'data/merged_shifts_wide_REST2.csv',
    'Rex_GaMD'  : 'data/merged_shifts_wide_Rex_GaMD.csv',
    'T-REMD'    : 'data/merged_shifts_wide_T-REMD.csv',
}

CORR_ATOMS        = ["C1'", "C4'", "C6", "C8"]
FLEXIBLE_RESIDUES = [23, 24, 25, 22, 40, 26, 39, 29, 36, 18, 44]

AFORM_HELIX_RESIDUES = [19, 43, 20, 42, 21, 41, 27, 38, 28, 37]
DOMAIN_DEFINITIONS = {
    "H1": [18, 19, 20, 21, 22, 40, 41, 42, 43, 44],
    "H2": [26, 27, 28, 29, 36, 37, 38, 39],
    "B":  [23, 24, 25],
}
DOMAIN_COLORS = {
    "H1": "#0072B2",
    "H2": "#009E73",
    "B":  "#D55E00",
}
DOMAINS_TO_ANALYZE = ["H1", "H2", "B"]

SPECIAL_CASES = {
    "N1/N3": {"N1": [18, 21, 26, 28, 36, 43], "N3": [38, 42]},
    "H1/H3": {"H1": [18, 21, 26, 28, 36, 43], "H3": [38, 42]},
}

# Muted qualitative palette — perceptually distinct across all 7 methods
# Change BAR_CMAP to switch back to a sequential colormap instead
BAR_CMAP        = None   # set to e.g. "Blues" to override METHOD_COLORS
BAR_SHADE_RANGE = (0.35, 0.92)

METHOD_COLORS = {
    'FARFAR':    '#6B8CAE',  # dusty blue
    'Alphafold': '#8FAF8F',  # dusty sage
    'cMD':       '#B08A6E',  # dusty sienna
    'GaMD':      '#9B8FAF',  # dusty mauve
    'REST2':     '#AFAA80',  # dusty khaki
    'Rex_GaMD':  '#7FAAAA',  # dusty teal
    'T-REMD':    '#AF8888',  # dusty rose
}

# =========================================================
# Color palette options for the weighted bar plots
#   Pick the one you like by setting PALETTE_CHOICE below.
# =========================================================
PALETTES = {
    "dusty": {   # current muted palette
        'FARFAR':'#6B8CAE', 'Alphafold':'#8FAF8F', 'cMD':'#B08A6E', 'GaMD':'#9B8FAF',
        'REST2':'#AFAA80', 'Rex_GaMD':'#7FAAAA', 'T-REMD':'#AF8888'},
    "okabe_ito": {   # colorblind-safe, vivid
        'FARFAR':'#0072B2', 'Alphafold':'#009E73', 'cMD':'#D55E00', 'GaMD':'#CC79A7',
        'REST2':'#E69F00', 'Rex_GaMD':'#56B4E9', 'T-REMD':'#444444'},
    "set2": {        # soft pastel, colorblind-friendly
        'FARFAR':'#66C2A5', 'Alphafold':'#FC8D62', 'cMD':'#8DA0CB', 'GaMD':'#E78AC3',
        'REST2':'#A6D854', 'Rex_GaMD':'#FFD92F', 'T-REMD':'#B3B3B3'},
    "tab10": {       # standard matplotlib vivid
        'FARFAR':'#1F77B4', 'Alphafold':'#FF7F0E', 'cMD':'#2CA02C', 'GaMD':'#D62728',
        'REST2':'#9467BD', 'Rex_GaMD':'#8C564B', 'T-REMD':'#E377C2'},
}
PALETTE_CHOICE = "dusty"   # <-- change to "okabe_ito", "set2", or "tab10"

ALIGN_CORR_LIMITS = True
SAVE_PNG = True
SAVE_TIF = False

# =========================================================
# Figure parameters
# =========================================================
PANEL_W  = 3.6
PANEL_H  = 3.2
WEIGHT_W = 4.8 * 1.25
WEIGHT_H = 3.0 * 1.25

# Larger axis fonts just for the weighted bar plots
WEIGHT_AXIS_LABEL = 6.3 * 2.5 * 1.25
WEIGHT_TICK_LABEL = 5.4 * 2.5 * 1.25
EXPORT_DPI = 600

BASE_FONT   = 5.6 * 2.5
AXIS_LABEL  = 6.3 * 2.5
TICK_LABEL  = 5.4 * 2.5
ANNOT_FONT  = 5.1 * 2.5
LEGEND_FONT = 4.8 * 2.5
PANEL_TAG   = 6.8 * 2.5

# =========================================================
# Figure style
# =========================================================
plt.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":          BASE_FONT,
    "axes.labelsize":     AXIS_LABEL,
    "axes.titlesize":     PANEL_TAG,
    "xtick.labelsize":    TICK_LABEL,
    "ytick.labelsize":    TICK_LABEL,
    "legend.fontsize":    LEGEND_FONT,
    "axes.linewidth":     1.2,
    "xtick.major.width":  1.0,
    "ytick.major.width":  1.0,
    "xtick.major.size":   3.5,
    "ytick.major.size":   3.5,
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
    "savefig.bbox":       "tight",
})

# =========================================================
# Helpers
# =========================================================
def safe_name(text: str) -> str:
    return text.replace("'", "").replace("/", "_").replace(" ", "_")

def calc_rmsd(calc, exp):
    calc = np.asarray(calc, dtype=float)
    exp  = np.asarray(exp,  dtype=float)
    if len(calc) == 0:
        return np.nan
    return np.sqrt(np.mean((calc - exp) ** 2))

def calc_r2(calc, exp):
    calc = np.asarray(calc, dtype=float)
    exp  = np.asarray(exp,  dtype=float)
    if len(calc) < 2:
        return np.nan
    r = np.corrcoef(calc, exp)[0, 1]
    return r ** 2

def linear_calibration(calc, exp):
    calc = np.asarray(calc, dtype=float)
    exp  = np.asarray(exp,  dtype=float)
    mask = np.isfinite(calc) & np.isfinite(exp)
    calc, exp = calc[mask], exp[mask]
    if len(calc) < 2:
        return calc, np.nan, np.nan
    m, b = np.polyfit(calc, exp, 1)
    return m * calc + b, m, b

def get_atom_type_filter(df, atom_type, special_cases=None):
    if special_cases is None:
        special_cases = SPECIAL_CASES
    if atom_type in special_cases:
        filters = [df["res"].isin(residues) & (df["atom"] == atom)
                   for atom, residues in special_cases[atom_type].items()]
        return np.logical_or.reduce(filters)
    return df["atom"] == atom_type

def load_and_merge_data(predicted_csv, measured_csv, n_conformers=20):
    dfp = pd.read_csv(predicted_csv).drop_duplicates()
    shift_cols = [f"shift_{i}" for i in range(1, n_conformers + 1)]
    dfp["shift_ens_avg"] = dfp[shift_cols].mean(axis=1)
    dfp["shift_ens_std"] = dfp[shift_cols].std(axis=1)
    dfp = dfp[["res", "atom", "shift_ens_avg", "shift_ens_std"]]

    dfm = pd.read_csv(measured_csv)
    dfm = dfm[["Resi", "Atom", "Shift", "SDev"]].rename(columns={
        "Resi": "res", "Atom": "atom",
        "Shift": "shift_nmr_avg", "SDev": "shift_nmr_std",
    })
    return dfm.merge(dfp, on=["res", "atom"], how="inner")

def create_domain_masks(df):
    return {domain: df["res"].isin(residues)
            for domain, residues in DOMAIN_DEFINITIONS.items()}

def create_domain_union(df, domain_masks):
    domain_union = np.zeros(len(df), dtype=bool)
    for domain in DOMAINS_TO_ANALYZE:
        if domain in domain_masks:
            domain_union |= domain_masks[domain]
    return domain_union

def compute_per_atom_metrics(df, residue_mask, special_cases=None, calibrate=True):
    if special_cases is None:
        special_cases = SPECIAL_CASES

    resonances = ["C1'", "C2'", "C3'", "C4'", "C5'", "C8", "C6", "C5", "C2", "N1/N3"]
    per_atom = {}

    for resonance in resonances:
        if resonance == "N1/N3":
            resonance_H  = "H1/H3"
            is_nitrogen  = True
        else:
            resonance_H  = "H" + resonance[1:]
            is_nitrogen  = resonance.startswith("N")

        filter_heavy = get_atom_type_filter(df, resonance, special_cases)
        filter_H     = get_atom_type_filter(df, resonance_H, special_cases)

        df_heavy = df.loc[residue_mask & filter_heavy,
                          ["shift_ens_avg", "shift_nmr_avg"]].dropna()
        if len(df_heavy) >= 2:
            calc, exp = df_heavy["shift_ens_avg"].values, df_heavy["shift_nmr_avg"].values
            calc_cal, m, b = linear_calibration(calc, exp) if calibrate else (calc, 1.0, 0.0)
            per_atom[resonance] = {
                "N": len(calc), "RMSD": calc_rmsd(calc_cal, exp),
                "R2": calc_r2(calc_cal, exp), "m": m, "b": b,
                "type": "15N" if is_nitrogen else "13C",
            }

        df_H = df.loc[residue_mask & filter_H,
                      ["shift_ens_avg", "shift_nmr_avg"]].dropna()
        if len(df_H) >= 2:
            calc_H, exp_H = df_H["shift_ens_avg"].values, df_H["shift_nmr_avg"].values
            calc_H_cal, m_H, b_H = linear_calibration(calc_H, exp_H) if calibrate else (calc_H, 1.0, 0.0)
            per_atom[resonance_H] = {
                "N": len(calc_H), "RMSD": calc_rmsd(calc_H_cal, exp_H),
                "R2": calc_r2(calc_H_cal, exp_H), "m": m_H, "b": b_H,
                "type": "1H",
            }

    return per_atom

def compute_N_weighted_metrics(per_atom_results):
    metrics = {k: {"N_total": 0, "sum_N_RMSD2": 0.0, "sum_N_R2": 0.0}
               for k in ["13C", "1H", "15N"]}

    for _, m in per_atom_results.items():
        atom_class = m.get("type")
        N, rmsd, r2 = m["N"], m["RMSD"], m["R2"]
        if atom_class in metrics and np.isfinite(rmsd) and np.isfinite(r2) and N > 0:
            metrics[atom_class]["N_total"]     += N
            metrics[atom_class]["sum_N_RMSD2"] += N * rmsd ** 2
            metrics[atom_class]["sum_N_R2"]    += N * r2

    out = {}
    for key in ["13C", "1H", "15N"]:
        N_total = metrics[key]["N_total"]
        if N_total > 0:
            out[f"{key}_N_weighted_RMSD"] = np.sqrt(metrics[key]["sum_N_RMSD2"] / N_total)
            out[f"{key}_N_weighted_R2"]   = metrics[key]["sum_N_R2"] / N_total
            out[f"{key}_N_total"]         = N_total
        else:
            out[f"{key}_N_weighted_RMSD"] = np.nan
            out[f"{key}_N_weighted_R2"]   = np.nan
            out[f"{key}_N_total"]         = 0
    return out

def compute_all_metrics(df, residue_mask):
    per_atom   = compute_per_atom_metrics(df, residue_mask,
                                          special_cases=SPECIAL_CASES, calibrate=True)
    n_weighted = compute_N_weighted_metrics(per_atom)
    return {"per_atom": per_atom, "N_weighted": n_weighted}

def get_atom_corr_data(df, atom_type, calibrate=True):
    sub = df.loc[get_atom_type_filter(df, atom_type, SPECIAL_CASES),
                 ["res", "shift_ens_avg", "shift_nmr_avg"]].dropna().copy()
    if len(sub) == 0:
        return sub
    calc, exp = sub["shift_ens_avg"].values, sub["shift_nmr_avg"].values
    if calibrate and len(sub) >= 2:
        calc_cal, _, _ = linear_calibration(calc, exp)
        sub["calc_plot"] = calc_cal
    else:
        sub["calc_plot"] = calc
    sub["exp_plot"] = exp
    return sub

def resolve_colors(method_names, palette_name=None):
    """Map each method to a color from the chosen named palette."""
    if palette_name is None:
        palette_name = PALETTE_CHOICE
    pal = PALETTES[palette_name]
    return {m: pal.get(m, "#888888") for m in method_names}

def save_figure(fig, out_base: Path):
    if SAVE_PNG:
        fig.savefig(out_base.with_suffix(".png"), dpi=EXPORT_DPI)
    if SAVE_TIF:
        fig.savefig(out_base.with_suffix(".tif"), dpi=EXPORT_DPI)

# =========================================================
# Load all data once
# =========================================================
all_data         = {}
all_domain_masks = {}
all_domain_union = {}

for method, pred_csv in ENSEMBLE_FILES.items():
    df   = load_and_merge_data(pred_csv, MEASURED_CSV, n_conformers=N_CONFORMERS)
    masks = create_domain_masks(df)
    all_data[method]         = df
    all_domain_masks[method] = masks
    all_domain_union[method] = create_domain_union(df, masks)
    print(f"Loaded {method}: {len(df)} merged rows")

# =========================================================
# Shared axis limits for correlation plots
# =========================================================
shared_limits = {}
if ALIGN_CORR_LIMITS:
    for atom in CORR_ATOMS:
        vals = []
        for method, df in all_data.items():
            sub = get_atom_corr_data(df, atom, calibrate=True)
            if len(sub) > 0:
                vals.extend(sub["calc_plot"].tolist())
                vals.extend(sub["exp_plot"].tolist())
        shared_limits[atom] = (min(vals) - 0.6, max(vals) + 0.6) if vals else None

# =========================================================
# Correlation figures (one per method × atom)
# =========================================================
for method, df in all_data.items():
    domain_masks = all_domain_masks[method]

    for atom in CORR_ATOMS:
        fig, ax = plt.subplots(figsize=(PANEL_W, PANEL_H))
        sub = get_atom_corr_data(df, atom, calibrate=True)
        all_calc, all_exp = [], []

        for domain in DOMAINS_TO_ANALYZE:
            mask_domain = domain_masks[domain]
            part = sub.loc[sub["res"].isin(df.loc[mask_domain, "res"])]
            if len(part) == 0:
                continue
            calc       = part["calc_plot"].values
            exp        = part["exp_plot"].values
            resnums    = part["res"].values
            color      = DOMAIN_COLORS[domain]
            aform_mask = np.isin(resnums, AFORM_HELIX_RESIDUES)

            if np.any(~aform_mask):
                ax.scatter(calc[~aform_mask], exp[~aform_mask],
                           s=48, color=color, edgecolors=color,
                           linewidth=0.8, alpha=0.85, zorder=3)
            if np.any(aform_mask):
                ax.scatter(calc[aform_mask], exp[aform_mask],
                           s=54, facecolors="none", edgecolors=color,
                           linewidth=1.2, alpha=0.90, zorder=4)

            all_calc.extend(calc.tolist())
            all_exp.extend(exp.tolist())

        if len(all_calc) > 0:
            all_calc = np.array(all_calc)
            all_exp  = np.array(all_exp)
            lo, hi   = (shared_limits[atom] if ALIGN_CORR_LIMITS
                                               and shared_limits.get(atom) is not None
                        else (min(all_calc.min(), all_exp.min()) - 0.6,
                              max(all_calc.max(), all_exp.max()) + 0.6))
            ax.plot([lo, hi], [lo, hi], "--", color="black", linewidth=1.0, zorder=2)
            ax.set_xlim(lo, hi)
            ax.set_ylim(lo, hi)
            rmsd = calc_rmsd(all_calc, all_exp)
            r2   = calc_r2(all_calc, all_exp)
            ax.text(0.97, 0.03, f"$R^2$={r2:.2f}\nRMSD={rmsd:.2f}",
                    transform=ax.transAxes, ha="right", va="bottom",
                    fontsize=ANNOT_FONT,
                    bbox=dict(boxstyle="round,pad=0.18", facecolor="white",
                              edgecolor="none", alpha=0.90))

        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("Calculated (ppm)", fontsize=AXIS_LABEL)
        ax.set_ylabel("Experimental (ppm)", fontsize=AXIS_LABEL)
        ax.tick_params(labelsize=TICK_LABEL, pad=2)
        fig.tight_layout()

        out_base = OUTPUT_DIR / f"figure_3_{safe_name(method)}_{safe_name(atom)}_correlation"
        save_figure(fig, out_base)
        plt.close(fig)
        print(f"Saved {out_base}")

# =========================================================
# Flexible-residue weighted metrics
# =========================================================
all_results_flex = {}
for method, df in all_data.items():
    flex_mask = df["res"].isin(FLEXIBLE_RESIDUES)
    all_results_flex[method] = compute_all_metrics(df, flex_mask)

comparison_flex = pd.DataFrame([
    {
        "Ensemble": method,
        "13C_RMSD": res["N_weighted"].get("13C_N_weighted_RMSD", np.nan),
        "13C_R2":   res["N_weighted"].get("13C_N_weighted_R2",   np.nan),
        "1H_RMSD":  res["N_weighted"].get("1H_N_weighted_RMSD",  np.nan),
        "1H_R2":    res["N_weighted"].get("1H_N_weighted_R2",    np.nan),
        "15N_RMSD": res["N_weighted"].get("15N_N_weighted_RMSD", np.nan),
        "15N_R2":   res["N_weighted"].get("15N_N_weighted_R2",   np.nan),
    }
    for method, res in all_results_flex.items()
]).set_index("Ensemble")

print("\nFlexible-residue weighted metrics:")
print(comparison_flex)

# =========================================================
# Weighted bar-plot  (two separate figures: RMSD and R2)
# =========================================================
def draw_weighted_metric(ax, comparison_df, metric, colors):
    """
    Draw a weighted-metric bar group onto a provided Axes.
    Nucleus type on x-axis, one bar per method.
    Returns (handles, labels) for building a legend.
    """
    methods_list = comparison_df.index.tolist()
    nuclei       = ["13C", "1H", "15N"]
    nuc_labels   = [r"$^{13}$C", r"$^{1}$H", r"$^{15}$N"]

    n_methods = len(methods_list)
    x         = np.arange(len(nuclei))
    gap       = 0.02
    width     = (0.78 - gap * (n_methods - 1)) / n_methods
    slot      = width + gap
    total     = n_methods * slot - gap

    # ── Subtle y-grid (behind bars) ───────────────────────────────────────────
    ax.yaxis.grid(True, color="0.88", linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)

    values_all = [
        comparison_df.loc[m, f"{nuc}_{metric}"]
        for m in methods_list for nuc in nuclei
        if np.isfinite(comparison_df.loc[m, f"{nuc}_{metric}"])
    ]
    ymax_data = max(values_all) if values_all else 1.0

    for j, method in enumerate(methods_list):
        offset = -total / 2 + j * slot + width / 2
        vals   = [comparison_df.loc[method, f"{nuc}_{metric}"] for nuc in nuclei]

        ax.bar(
            x + offset, vals, width=width,
            color=colors[method], edgecolor="white", linewidth=0.5,
            label=method, zorder=3,
        )

        # Value annotations (rotated so they stay legible between narrow bars)
        ypad = 0.015 if metric == "R2" else 0.018 * ymax_data
        for xi, yi in zip(x + offset, vals):
            if np.isfinite(yi):
                ax.text(xi, yi + ypad, f"{yi:.2f}",
                        ha="center", va="bottom",
                        fontsize=ANNOT_FONT * 0.6,
                        color="#1a1a1a", rotation=90)

    # ── Axis labels & ticks ───────────────────────────────────────────────────
    ax.set_xticks(x)
    ax.set_xticklabels(nuc_labels, fontsize=WEIGHT_TICK_LABEL)
    ax.tick_params(axis="y", labelsize=WEIGHT_TICK_LABEL, pad=2)
    ax.set_xlabel("Nucleus type", fontsize=WEIGHT_AXIS_LABEL)

    if metric == "R2":
        ax.set_ylabel(r"N-weighted $R^2$", fontsize=WEIGHT_AXIS_LABEL)
        ax.set_ylim(0, max(1.05, ymax_data + 0.14))
    else:
        ax.set_ylabel("N-weighted RMSD (ppm)", fontsize=WEIGHT_AXIS_LABEL)
        ax.set_ylim(0, ymax_data * 1.28)

    # ── Spine cleanup ─────────────────────────────────────────────────────────
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for sp in ["left", "bottom"]:
        ax.spines[sp].set_linewidth(1.1)
        ax.spines[sp].set_color("0.20")
    ax.tick_params(axis="both", width=1.1, length=4, color="0.20", direction="out")

    return ax.get_legend_handles_labels()


def plot_weighted_metric(comparison_df, metric="RMSD", palette_name=None):
    """
    Single self-contained figure (with legend) for one metric.
    Sized to WEIGHT_W x WEIGHT_H so two of them tile neatly into one row.
    """
    colors = resolve_colors(comparison_df.index.tolist(), palette_name)

    fig, ax = plt.subplots(figsize=(WEIGHT_W, WEIGHT_H))
    handles, labels = draw_weighted_metric(ax, comparison_df, metric, colors)

    leg = ax.legend(
        handles, labels,
        ncol=4, frameon=True,
        loc="lower center", bbox_to_anchor=(0.5, 1.01),
        borderpad=0.5, handletextpad=0.4, columnspacing=0.9,
        handlelength=1.2, fontsize=LEGEND_FONT,
    )
    leg.get_frame().set_edgecolor("0.75")
    leg.get_frame().set_linewidth(0.7)
    leg.get_frame().set_alpha(0.92)

    fig.tight_layout()
    return fig

# =========================================================
# Save the two bar plots (assemble into one row yourself)
# =========================================================
fig_rmsd = plot_weighted_metric(comparison_flex, metric="RMSD")
save_figure(fig_rmsd, OUTPUT_DIR / "figure_3_flexible_residues_weighted_RMSD")
plt.close(fig_rmsd)
print(f"Saved {OUTPUT_DIR / 'flexible_residues_weighted_RMSD'}")

fig_r2 = plot_weighted_metric(comparison_flex, metric="R2")
save_figure(fig_r2, OUTPUT_DIR / "figure_3_flexible_residues_weighted_R2")
plt.close(fig_r2)
print(f"Saved {OUTPUT_DIR / 'flexible_residues_weighted_R2'}")

print("\nDone.")

Loaded FARFAR: 502 merged rows
Loaded Alphafold: 502 merged rows
Loaded cMD: 502 merged rows
Loaded GaMD: 502 merged rows
Loaded REST2: 502 merged rows
Loaded Rex_GaMD: 502 merged rows
Loaded T-REMD: 502 merged rows
Saved outputs/figure_3_FARFAR_C1_correlation
Saved outputs/figure_3_FARFAR_C4_correlation
Saved outputs/figure_3_FARFAR_C6_correlation
Saved outputs/figure_3_FARFAR_C8_correlation
Saved outputs/figure_3_Alphafold_C1_correlation
Saved outputs/figure_3_Alphafold_C4_correlation
Saved outputs/figure_3_Alphafold_C6_correlation
Saved outputs/figure_3_Alphafold_C8_correlation
Saved outputs/figure_3_cMD_C1_correlation
Saved outputs/figure_3_cMD_C4_correlation
Saved outputs/figure_3_cMD_C6_correlation
Saved outputs/figure_3_cMD_C8_correlation
Saved outputs/figure_3_GaMD_C1_correlation
Saved outputs/figure_3_GaMD_C4_correlation
Saved outputs/figure_3_GaMD_C6_correlation
Saved outputs/figure_3_GaMD_C8_correlation
Saved outputs/figure_3_REST2_C1_correlation
Saved outputs/figure_3_REST2